一个tokenizer网址：https://tiktokenizer.vercel.app/
从视频中可以看到，以GPT2为例，数学加法中的数字可能会被拆分；同一单词可能因为大小写不同而被以不一样的方式拆分；非英文文本token数被拉长了，有许多小的边界，是由于训练数据中英文文本占多数，与用于tokenizer和分词本身的训练有关
而对于GPT4的tokenizer，其token数是GPT2的两倍（embedding table也会变大，后续的softmax会增加计算量），那么同样的文本其token表示可以被压缩到一半

ord()只接受单个字符，不能直接就用这个来进行tokenize，原因：

1、过长

2、Unicode一直变化，不是稳定表示

In [1]:
[ord(x) for x in "안녕하세요 👋 (hello in Korean!)"]

[50504,
 45397,
 54616,
 49464,
 50836,
 32,
 128075,
 32,
 40,
 104,
 101,
 108,
 108,
 111,
 32,
 105,
 110,
 32,
 75,
 111,
 114,
 101,
 97,
 110,
 33,
 41]

直接使用UTF-8也不好，因为这会将输入文本变为很长的字节序列，但transformer的上下文长度有限，不利于关注本身的输入文本（都集中在了字节序列上）。

所希望的是能够支持更大的词汇量，并将其作为超参进行调整

In [2]:
list("안녕하세요 👋 (hello in Korean!)".encode("utf-8"))

[236,
 149,
 136,
 235,
 133,
 149,
 237,
 149,
 152,
 236,
 132,
 184,
 236,
 154,
 148,
 32,
 240,
 159,
 145,
 139,
 32,
 40,
 104,
 101,
 108,
 108,
 111,
 32,
 105,
 110,
 32,
 75,
 111,
 114,
 101,
 97,
 110,
 33,
 41]

### Byte pair encoding

找到出现频率最高的两个token对，一旦确定了这个对，就用其替代所有文本中的这个对，重复这个过程直到达到预定的词汇量

https://en.wikipedia.org/wiki/Byte-pair_encoding

In [1]:
# text from https://www.reedbeta.com/blog/programmers-intro-to-unicode/
text = "Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception."
tokens = text.encode("utf-8") # raw bytes
tokens = list(map(int, tokens)) # convert to a list of integers in range 0..255 for convenience
print('---')
print(text)
print("length:", len(text))
print('---')
print(tokens)
print("length:", len(tokens))

---
Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception.
length: 533
---
[239, 188, 181, 239, 189, 142, 239, 189, 137, 239, 189, 131, 239, 189, 143, 239, 189, 132, 239, 189, 133, 33, 32, 240, 159, 133, 164, 240, 159, 133, 157, 240, 159, 133, 152, 240, 159, 133, 146, 240, 159, 133, 158, 240, 159, 133, 147, 240, 159, 133, 148, 226, 128, 189, 32, 240, 159, 135, 186, 226, 128, 140, 240, 159, 135, 179, 226, 128, 140, 240, 159, 135, 174, 226, 128, 140, 240, 159, 135, 168, 226, 128, 140, 240, 159, 135, 180, 226, 128, 140

操作：找最频繁的字节对，合并

zip函数作用：将多个可迭代对象按位置配对打包，返回迭代器。例如;

'''python
a = [1, 2, 3]
b = ['a', 'b', 'c']

list(zip(a, b))
# 结果: [(1, 'a'), (2, 'b'), (3, 'c')]

In [ ]:
def get_stats(ids):
    count = {}
    for pair in zip(ids, ids[1:]):  # 通过切片错位，实现相邻元素配对
        count[pair] = count.get(pair, 0) + 1
    return count

stats = get_stats(tokens)

In [ ]:
top_pair = max(stats, key=stats.get)  # 字典的 .get() 方法，用于获取键对应的值
top_pair

In [1]:
def merge(ids, pair, idx):
  # in the list of ints (ids), replace all consecutive occurences of pair with the new token idx
  newids = []
  i = 0
  while i < len(ids):
    # if we are not at the very last position AND the pair matches, replace it
    if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
      newids.append(idx)
      i += 2
    else:
      newids.append(ids[i])
      i += 1
  return newids

print(merge([5, 6, 6, 7, 9, 1], (6, 7), 99))

tokens2 = merge(tokens, top_pair, 256)
print(tokens2)
print("length:", len(tokens2))

[5, 6, 99, 9, 1]


NameError: name 'tokens' is not defined

至于词汇表大小，是一个超参数，由自己进行设置和平衡

In [ ]:
# ---
vocab_size = 276 # 刚好进行20次merge，256 + 20 = 276
num_merges = vocab_size - 256
ids = list(tokens) # copy so we don't destroy the original list

merges = {} # (int, int) -> int
for i in range(num_merges):
  stats = get_stats(ids)
  pair = max(stats, key=stats.get)
  idx = 256 + i
  print(f"merging {pair} into a new token {idx}")
  ids = merge(ids, pair, idx)
  merges[pair] = idx

## decoding

In [ ]:
vocab = {idx:byttes([idx]) for idx in range(256)} # 0..255 are bytes
for (p0, p1), idx in merges.items():
  vocab[idx] = vocab[p0] + vocab[p1]